# ARTI 308 -- Lab 5: Feature Engineering (Classification)
## Heart Disease Prediction using Feature Engineering

### Lab Focus
This dataset is already clean (no missing values, no duplicate rows, consistent data types).
In this lab, we focus on **feature engineering** for a classification task -- predicting whether a patient has heart disease.

- **Problem Type:** Binary Classification
- **Target Variable:** `target` (0 = No Disease, 1 = Disease)
- **Dataset Source:** [UCI Heart Disease](https://archive.ics.uci.edu/ml/datasets/heart+Disease)

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

sns.set(style="whitegrid")

## 2. Load the Dataset

In [ ]:
df = pd.read_csv("heart.csv")
df.head(10)

The first rows confirm the dataset loaded correctly. Each row represents a patient record with medical attributes and a heart disease diagnosis in the `target` column.

## 3. Quick Dataset Checks (Cleanliness Confirmation)

In [ ]:
print("Shape:", df.shape)
print("\nMissing values per column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

The dataset is clean -- no missing values and no duplicate rows. This means we can skip data cleaning and spend our effort on feature engineering instead.

## 4. Target Variable and Class Balance

In [ ]:
print(df['target'].value_counts())

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df)
plt.title("Heart Disease Distribution")
plt.xlabel("Target (0=No Disease, 1=Disease)")
plt.ylabel("Count")
plt.show()

The classes are roughly balanced. When classes are imbalanced, the model may favor the dominant class and appear accurate while performing poorly on the minority class. In such cases, the confusion matrix and per-class metrics (precision, recall) are more informative than overall accuracy alone.

## 5. Identify Feature Types

In [ ]:
df.dtypes

In [ ]:
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
id_cols = []  # No ID columns in this dataset
target_col = 'target'

print(f"Numerical features ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")
print(f"Target: {target_col}")

All features are stored as integers, but several are actually categorical (coded as numbers). For example, `cp` (chest pain type) takes values 0-3 representing distinct categories, not a numeric scale. Identifying feature types correctly is crucial before applying transformations and engineering new features.

## 6. Feature Engineering

### 6.1 Age Groups

Bin continuous age into meaningful medical age groups.

In [ ]:
df['age_group'] = pd.cut(df['age'], bins=[0, 40, 55, 70, 100], labels=['Young', 'Middle', 'Senior', 'Elderly'])
df[['age', 'age_group']].head(10)

Binning transforms a continuous feature into a categorical one. Age groups have clinical significance -- heart disease risk increases with age brackets, and physicians often reason about patients in terms of age ranges rather than exact ages.

### 6.2 Blood Pressure Risk Category

Create a risk category from resting blood pressure based on medical standards.

In [ ]:
def bp_category(bp):
    if bp < 120:
        return 'Normal'
    elif bp < 140:
        return 'Elevated'
    else:
        return 'High'

df['bp_risk'] = df['trestbps'].apply(bp_category)
df[['trestbps', 'bp_risk']].head(10)

Medical thresholds provide domain-informed feature engineering. These blood pressure categories (Normal < 120, Elevated 120-139, High 140+) carry more predictive meaning than raw values in many cases, because the relationship between BP and disease risk is not strictly linear.

### 6.3 Cholesterol Risk Level

Categorize cholesterol levels based on medical guidelines.

In [ ]:
def chol_category(chol):
    if chol < 200:
        return 'Desirable'
    elif chol < 240:
        return 'Borderline'
    else:
        return 'High'

df['chol_risk'] = df['chol'].apply(chol_category)
df[['chol', 'chol_risk']].head(10)

Clinical cholesterol categories simplify the model's job -- instead of learning exact thresholds from the data, the model can leverage established medical categories (Desirable < 200, Borderline 200-239, High 240+) that are already known to correlate with heart disease risk.

### 6.4 Heart Rate Reserve

Create a feature measuring the difference between max heart rate achieved and a typical resting heart rate.

In [ ]:
df['hr_reserve'] = df['thalach'] - 72  # 72 bpm is average resting heart rate
df[['thalach', 'hr_reserve']].head(10)

Heart rate reserve indicates cardiovascular fitness. A higher reserve suggests better heart function -- the heart can increase its rate significantly during exercise, which is a positive sign of cardiac health.

### 6.5 Exercise Risk Score

Create a composite feature combining exercise-related indicators.

In [ ]:
df['exercise_risk'] = df['exang'] + df['oldpeak'] + (2 - df['slope'])
df[['exang', 'oldpeak', 'slope', 'exercise_risk']].head(10)

Composite features combine related signals into a single indicator. This exercise risk score aggregates exercise-induced angina (`exang`), ST depression (`oldpeak`), and the slope of the ST segment into one value. Higher scores indicate greater exercise-related cardiac risk.

### 6.6 Summary of Engineered Features

In [ ]:
print("Original features:", 13)
print("Engineered features: age_group, bp_risk, chol_risk, hr_reserve, exercise_risk")
print("Total columns now:", df.shape[1])
df.head()

## 7. Encoding Categorical Features

Machine learning models require numerical input. Categorical features -- including our newly engineered ones like `age_group`, `bp_risk`, and `chol_risk` -- must be encoded. We use One-Hot Encoding via scikit-learn's `ColumnTransformer` inside a `Pipeline` to keep preprocessing and modeling together.

In [ ]:
# Update feature lists with engineered features
new_numerical = numerical_cols + ['hr_reserve', 'exercise_risk']
new_categorical = categorical_cols + ['age_group', 'bp_risk', 'chol_risk']

print(f"Numerical features for model ({len(new_numerical)}): {new_numerical}")
print(f"Categorical features for model ({len(new_categorical)}): {new_categorical}")

## 8. Prepare Data for Modeling

In [ ]:
X = df[new_numerical + new_categorical]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

The stratified split ensures both training and testing sets have similar proportions of each class (Disease vs No Disease). This is important for classification tasks to get reliable performance estimates.

## 9. Build Pipeline with Encoding

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), new_numerical),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), new_categorical)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

print("Pipeline created successfully")
print(pipeline)

The pipeline automates preprocessing and model training into a single object. The `ColumnTransformer` applies different transformations to numerical vs categorical features: `StandardScaler` normalizes numerical features to mean=0 and std=1, while `OneHotEncoder` converts categorical features into binary columns.

## 10. Train the Model

In [ ]:
pipeline.fit(X_train, y_train)
print("Model trained successfully")

## 11. Evaluate the Model

In [ ]:
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

The confusion matrix shows four outcomes:
- **True Negatives (top-left):** Correctly predicted No Disease
- **False Positives (top-right):** Incorrectly predicted Disease (healthy patient flagged)
- **False Negatives (bottom-left):** Incorrectly predicted No Disease (missed diagnosis)
- **True Positives (bottom-right):** Correctly predicted Disease

In medical contexts, **false negatives** (missing a disease) are especially costly because a patient with heart disease would go untreated. We want to minimize false negatives even if it means accepting more false positives.

## 12. Feature Importance

In [ ]:
# Get feature names after preprocessing
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

importances = pipeline.named_steps['classifier'].feature_importances_
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(15))
plt.title("Top 15 Feature Importances")
plt.tight_layout()
plt.show()

Feature importance shows which features the model relies on most for predictions. This helps validate whether our engineered features contribute meaningfully. If engineered features appear among the top contributors, it confirms that domain knowledge can improve model performance.

## 13. Compare: With vs Without Engineered Features

In [ ]:
# Model WITHOUT engineered features
X_original = df[numerical_cols + categorical_cols]
X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(
    X_original, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor_orig = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

pipeline_orig = Pipeline(steps=[
    ('preprocessor', preprocessor_orig),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline_orig.fit(X_train_orig, y_train_orig)
y_pred_orig = pipeline_orig.predict(X_test_orig)

print("Accuracy WITHOUT engineered features:", accuracy_score(y_test_orig, y_pred_orig))
print("Accuracy WITH engineered features:   ", accuracy_score(y_test, y_pred))

The comparison shows whether feature engineering improved the model. Even if the improvement is small, the exercise demonstrates the process of applying domain knowledge to create meaningful features. In practice, feature engineering is often the most impactful step in an ML pipeline.

# Assignment

In this assignment, you will practice feature engineering:

- **Task 1:** Create at least two additional engineered features based on domain knowledge.
- **Task 2:** Add the new features to the pipeline and retrain the model.
- **Task 3:** Compare model performance before and after your new features.
- **Task 4:** Analyze feature importance and identify the top contributing features.

End of Lab 5.